# LeetCode 1276 — Number of Burgers with No Waste of Ingredients bằng Python

Link đề: https://leetcode.com/problems/number-of-burgers-with-no-waste-of-ingredients/ · Độ khó: **Medium** · Chủ đề: Math

Skeleton LeetCode:

```python
class Solution:
    def numOfBurgers(self, tomatoSlices: int, cheeseSlices: int) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Biết **mô hình hoá** một bài đố chữ thành **hệ phương trình tuyến tính** hai ẩn với điều kiện nghiệm nguyên không âm.
2. Đi từ **vét cạn hai ẩn** → **vét cạn một ẩn** → **binary search trên ẩn** → **công thức đóng `O(1)`**.
3. Hiểu trực giác "**bắt đầu toàn burger nhỏ, đổi dần sang burger lớn**" — mỗi lần đổi thêm đúng 2 lát cà chua.
4. Nắm vững phép chia nguyên trong Python: `//`, `%`, `divmod`, và vì sao không dùng `/`.
5. Tránh các lỗi kinh điển: quên kiểm tra **tính chẵn**, quên kiểm tra **không âm**, trả về số thực.

## 2. Đề bài & ví dụ

Cho hai số nguyên `tomatoSlices` (số lát cà chua) và `cheeseSlices` (số lát phô mai). Có hai loại burger:

- **Jumbo Burger:** 4 lát cà chua + 1 lát phô mai.
- **Small Burger:** 2 lát cà chua + 1 lát phô mai.

Trả về `[total_jumbo, total_small]` sao cho **dùng hết** cả cà chua lẫn phô mai (không thừa lát nào). Nếu không thể, trả về `[]`.

**Ví dụ 1:** `tomatoSlices = 16, cheeseSlices = 7` → `[1, 6]` (vì `4·1 + 2·6 = 16` cà chua và `1 + 6 = 7` phô mai).

**Ví dụ 2:** `tomatoSlices = 17, cheeseSlices = 4` → `[]`.

**Ví dụ 3:** `tomatoSlices = 4, cheeseSlices = 17` → `[]` (1 jumbo thì thừa 16 phô mai, 2 small thì thừa 15 phô mai).

**Ràng buộc:** `0 <= tomatoSlices, cheeseSlices <= 10^7`.

In [1]:
import bisect
import random
import time
from typing import List

examples = [(16, 7, [1, 6]), (17, 4, []), (4, 17, [])]
for t, c, want in examples:
    print(f"tomato={t:>2}, cheese={c:>2} -> mong đợi {want}")

tomato=16, cheese= 7 -> mong đợi [1, 6]
tomato=17, cheese= 4 -> mong đợi []
tomato= 4, cheese=17 -> mong đợi []


## 3. Kiến thức nền

### 3.1 Chia nguyên `//`, chia lấy dư `%`, `divmod`

`a // b` là thương nguyên (làm tròn **xuống**), `a % b` là số dư. `divmod(a, b)` trả cả hai. Số chẵn ⇔ `a % 2 == 0`.

In [2]:
print(17 // 2, 17 % 2, divmod(17, 2))
print(16 // 2, 16 % 2, divmod(16, 2))
print("chú ý số âm:", -7 // 2, -7 % 2)   # làm tròn xuống: -4, dư 1

8 1 (8, 1)
8 0 (8, 0)
chú ý số âm: -4 1


### 3.2 Phép chia `/` luôn cho số thực

`16 / 2` cho `8.0` (kiểu `float`), không phải `8`. Đề yêu cầu trả số nguyên ⇒ dùng `//` sau khi đã chắc chắn chia hết.

In [3]:
x = 16 / 2
print(x, type(x).__name__)
y = 16 // 2
print(y, type(y).__name__)
print("17 / 2 =", 17 / 2, "-> không phải số burger hợp lệ")

8.0 float
8 int
17 / 2 = 8.5 -> không phải số burger hợp lệ


### 3.3 Hệ phương trình hai ẩn

Gọi `x` = số jumbo, `y` = số small:

```
4x + 2y = T     (cà chua)
 x +  y = C     (phô mai)
```

Khử `y`: lấy phương trình 1 trừ 2 lần phương trình 2 ⇒ `2x = T − 2C` ⇒ `x = (T − 2C) / 2`. Thay vào ⇒ `y = C − x = (4C − T) / 2`.

In [4]:
T, C = 16, 7
x = (T - 2 * C) / 2
y = (4 * C - T) / 2
print("x =", x, "| y =", y, "| kiểm tra:", 4 * x + 2 * y, x + y)

x = 1.0 | y = 6.0 | kiểm tra: 16.0 7.0


### 3.4 Trả về danh sách rỗng `[]`

Trong Python, `[]` là một list hợp lệ, có độ dài 0 và có giá trị chân lý `False`. Hàm trả `[]` khi không có nghiệm.

In [5]:
res = []
print(len(res), bool(res), res == [])
print(bool([1, 6]))

0 False True
True


### 3.5 Binary search với `bisect`

`bisect.bisect_left(a, v)` tìm vị trí đầu tiên mà `a[i] >= v` trong dãy tăng. Từ Python 3.10, có tham số `key` để tìm trên một hàm đơn điệu mà **không cần tạo mảng**: truyền `range(...)` làm `a`.

In [6]:
a = [1, 3, 3, 5, 8]
print(bisect.bisect_left(a, 3), bisect.bisect_left(a, 4))
# tìm x nhỏ nhất trong [0, 100] sao cho x*x >= 50
print(bisect.bisect_left(range(101), 50, key=lambda x: x * x))

1 3
8


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** (`T = 16`, `C = 7`):

- Mỗi burger dùng đúng **1** lát phô mai ⇒ tổng số burger **bắt buộc** là `C = 7`.
- Nếu cả 7 đều là small: cần `2·7 = 14` cà chua. Còn thừa `16 − 14 = 2` lát.
- Đổi 1 small thành 1 jumbo: phô mai không đổi, cà chua tăng thêm `4 − 2 = 2`. Cần thêm 2 lát ⇒ đổi **1** cái.
- Kết quả: `1` jumbo, `6` small. ✔

Tổng quát: số cà chua dùng được nằm trong đoạn `[2C, 4C]` (toàn small → toàn jumbo) và **chỉ tăng theo bước 2**. Vậy có nghiệm ⇔ `T` chẵn và `2C ≤ T ≤ 4C`.

- **Ví dụ 2** (`T = 17`): lẻ ⇒ không có nghiệm.
- **Ví dụ 3** (`T = 4`, `C = 17`): cần ít nhất `2·17 = 34` cà chua, mà chỉ có 4 ⇒ không có nghiệm.

In [7]:
for t, c, want in examples:
    lo, hi = 2 * c, 4 * c
    print(f"T={t}, C={c}: đoạn cà chua dùng được [{lo}, {hi}], T chẵn? {t % 2 == 0}, "
          f"trong đoạn? {lo <= t <= hi}")

T=16, C=7: đoạn cà chua dùng được [14, 28], T chẵn? True, trong đoạn? True
T=17, C=4: đoạn cà chua dùng được [8, 16], T chẵn? False, trong đoạn? False
T=4, C=17: đoạn cà chua dùng được [34, 68], T chẵn? True, trong đoạn? False


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử mọi cặp `(x, y)` với `0 ≤ x ≤ T/4`, `0 ≤ y ≤ T/2`, kiểm tra hai phương trình. Đây là cách "không cần suy nghĩ", chỉ để có đáp án chắc chắn đúng trên dữ liệu nhỏ.

In [8]:
def burgers_brute2(T, C):
    for x in range(T // 4 + 1):
        for y in range(T // 2 + 1):
            if 4 * x + 2 * y == T and x + y == C:
                return [x, y]
    return []

for t, c, want in examples:
    print(burgers_brute2(t, c), "| mong đợi", want)

[1, 6] | mong đợi [1, 6]
[] | mong đợi []
[] | mong đợi []


**Độ phức tạp:** `O(T²)` — với `T = 10^7` là `~10^13` phép thử, hoàn toàn không khả thi.

**Nút thắt:** ta đang thử `y` một cách "mù quáng", trong khi phương trình phô mai đã **xác định** `y = C − x` ngay khi biết `x`. Một ẩn là thừa.

### Cách 1b — Vét cạn một ẩn

Duyệt `x = 0 … C`, đặt `y = C − x` (phương trình phô mai luôn đúng), chỉ cần kiểm tra cà chua.

In [9]:
def burgers_brute1(T, C):
    for x in range(C + 1):            # chú ý C + 1: x có thể bằng C
        y = C - x
        if 4 * x + 2 * y == T:
            return [x, y]
    return []

for t, c, want in examples:
    print(burgers_brute1(t, c), "| mong đợi", want)

t0 = time.perf_counter()
print(burgers_brute1(4 * 10**6 - 2, 10**6), f"{time.perf_counter() - t0:.3f}s với C = 10^6")

[1, 6] | mong đợi [1, 6]
[] | mong đợi []
[] | mong đợi []
[999999, 1] 0.104s với C = 10^6


`O(C)` thời gian, `O(1)` bộ nhớ. Với `C = 10^7` thì khoảng 10^7 vòng lặp Python (~1–2 giây) — có thể quá thời gian. Ta cần khai thác thêm cấu trúc.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — tính đơn điệu:** khi `y = C − x`, lượng cà chua dùng là `f(x) = 4x + 2(C − x) = 2C + 2x`, **tăng ngặt** theo `x`. Hàm tăng ⇒ tìm `x` bằng **binary search**.

**Insight 2 — giải luôn:** `f(x) = T` ⇔ `x = (T − 2C) / 2`. Không cần tìm.

**Quy tắc "nếu… thì…":**
- Nếu `T` lẻ ⇒ `[]` (vế trái `4x + 2y` luôn chẵn).
- Nếu `T < 2C` ⇒ quá ít cà chua kể cả khi toàn small ⇒ `[]` (vì `x` sẽ âm).
- Nếu `T > 4C` ⇒ quá nhiều cà chua kể cả khi toàn jumbo ⇒ `[]` (vì `y` sẽ âm).
- Ngược lại ⇒ `x = T // 2 − C`, `y = C − x` (hoặc `y = 2C − T // 2`).

**Invariant** khi "đổi dần": luôn có `x + y = C` và cà chua dùng `= 2C + 2x`.

## 7. Các cách tiếp theo

### Cách 2 — Bắt đầu toàn small, đổi dần sang jumbo (mô phỏng có kiểm soát)

**Từng đoạn:**
1. Đặt `x = 0`, `y = C`, `used = 2C`.
2. Nếu `used > T` ⇒ đã quá ⇒ `[]`.
3. Phần thiếu `need = T − used`. Mỗi lần đổi bù được 2 lát ⇒ cần `need / 2` lần đổi; phải chia hết và không vượt quá `C`.

Đây chính là công thức đóng nhưng được giải thích theo hướng "kể chuyện". Ta cũng viết bản mô phỏng từng bước (chậm) để dễ hình dung.

In [10]:
def burgers_convert_slow(T, C):
    x, y, used = 0, C, 2 * C
    while used < T and y > 0:          # còn thiếu cà chua và còn small để đổi
        x, y, used = x + 1, y - 1, used + 2
    return [x, y] if used == T else []

def burgers_convert(T, C):
    used = 2 * C
    need = T - used
    if need < 0 or need % 2:
        return []
    x = need // 2
    if x > C:
        return []
    return [x, C - x]

for t, c, want in examples:
    print(burgers_convert_slow(t, c), burgers_convert(t, c), "| mong đợi", want)

[1, 6] [1, 6] | mong đợi [1, 6]
[] [] | mong đợi []
[] [] | mong đợi []


**Độ phức tạp:** bản mô phỏng `O(C)`; bản tính trực tiếp `O(1)`.

### Cách 3 — Binary search trên số jumbo `x`

**Từng đoạn:**
1. Không gian tìm kiếm `x ∈ [0, C]`.
2. `f(x) = 2C + 2x` tăng ⇒ tìm `x` nhỏ nhất với `f(x) ≥ T` bằng `lo, hi`.
3. Kiểm tra `x ≤ C` và `f(x) == T`.

In [11]:
def burgers_binary(T, C):
    lo, hi = 0, C
    while lo < hi:                      # tìm x nhỏ nhất có 2C + 2x >= T
        mid = (lo + hi) // 2
        if 2 * C + 2 * mid >= T:
            hi = mid
        else:
            lo = mid + 1
    x = lo
    return [x, C - x] if 4 * x + 2 * (C - x) == T else []

for t, c, want in examples:
    print(burgers_binary(t, c), "| mong đợi", want)

[1, 6] | mong đợi [1, 6]
[] | mong đợi []
[] | mong đợi []


Biến thể dùng thư viện `bisect` (Python ≥ 3.10) — ngắn gọn hơn:

In [12]:
def burgers_bisect(T, C):
    x = bisect.bisect_left(range(C + 1), T, key=lambda k: 2 * C + 2 * k)
    if x <= C and 2 * C + 2 * x == T:
        return [x, C - x]
    return []

for t, c, want in examples:
    print(burgers_bisect(t, c), "| mong đợi", want)

[1, 6] | mong đợi [1, 6]
[] | mong đợi []
[] | mong đợi []


Code nộp LeetCode cho Cách 3:

In [13]:
class SolutionBinary:
    def numOfBurgers(self, tomatoSlices: int, cheeseSlices: int) -> List[int]:
        T, C = tomatoSlices, cheeseSlices
        lo, hi = 0, C
        while lo < hi:
            mid = (lo + hi) // 2
            if 2 * C + 2 * mid >= T:
                hi = mid
            else:
                lo = mid + 1
        return [lo, C - lo] if 2 * C + 2 * lo == T else []

print([SolutionBinary().numOfBurgers(t, c) for t, c, _ in examples])

[[1, 6], [], []]


**Độ phức tạp Cách 3:** `O(log C)` thời gian, `O(1)` bộ nhớ.

### Cách 4 — Công thức đóng từ hệ phương trình

**Từng đoạn:**
1. `x = (T − 2C) / 2`, `y = (4C − T) / 2`.
2. Nghiệm hợp lệ ⇔ cả hai **nguyên** và **không âm** ⇔ `T` chẵn, `T ≥ 2C`, `T ≤ 4C`.
3. Dùng `//` sau khi đã chắc `T` chẵn ⇒ kết quả là `int`.

In [14]:
def burgers_formula(T, C):
    if T % 2 == 1:
        return []
    x = T // 2 - C          # = (T - 2C) / 2
    y = 2 * C - T // 2      # = (4C - T) / 2
    if x < 0 or y < 0:
        return []
    return [x, y]

for t, c, want in examples:
    print(burgers_formula(t, c), "| mong đợi", want)

[1, 6] | mong đợi [1, 6]
[] | mong đợi []
[] | mong đợi []


In [15]:
class SolutionFormula:
    def numOfBurgers(self, tomatoSlices: int, cheeseSlices: int) -> List[int]:
        T, C = tomatoSlices, cheeseSlices
        if T % 2 or T < 2 * C or T > 4 * C:
            return []
        x = (T - 2 * C) // 2
        return [x, C - x]

print([SolutionFormula().numOfBurgers(t, c) for t, c, _ in examples])

[[1, 6], [], []]


**Độ phức tạp Cách 4:** `O(1)` thời gian và bộ nhớ.

### Vì sao nghiệm (nếu có) là **duy nhất**?

Hai phương trình độc lập tuyến tính (định thức `4·1 − 2·1 = 2 ≠ 0`) ⇒ hệ có đúng một nghiệm thực. Vì vậy không cần lo "chọn đáp án nào" — nếu nghiệm thực đó nguyên và không âm thì đó là đáp án, còn không thì `[]`.

In [16]:
# Kiểm chứng tính duy nhất bằng vét cạn: đếm số nghiệm cho mọi (T, C) nhỏ
max_count = 0
for T in range(0, 61):
    for C in range(0, 31):
        cnt = sum(1 for x in range(C + 1) if 4 * x + 2 * (C - x) == T)
        max_count = max(max_count, cnt)
print("số nghiệm tối đa trên mọi (T, C) nhỏ:", max_count)

số nghiệm tối đa trên mọi (T, C) nhỏ: 1


## 8. TRACE từng bước

**Trace Cách 2 (đổi dần)** với `T = 16, C = 7`:

| bước | x (jumbo) | y (small) | cà chua dùng | so với T=16 |
|---|---|---|---|---|
| 0 | 0 | 7 | 14 | thiếu 2 |
| 1 | 1 | 6 | 16 | **bằng** → dừng |

**Trace Cách 3 (binary search)** với `T = 26, C = 9` (đáp án `[4, 5]`): `f(x) = 18 + 2x`.

| lo | hi | mid | f(mid) | f(mid) ≥ 26? | cập nhật |
|---|---|---|---|---|---|
| 0 | 9 | 4 | 26 | có | hi = 4 |
| 0 | 4 | 2 | 22 | không | lo = 3 |
| 3 | 4 | 3 | 24 | không | lo = 4 |
| 4 | 4 | — | — | — | dừng, x = 4 |

In [17]:
def trace_convert(T, C):
    x, y, used = 0, C, 2 * C
    print(f"bắt đầu: x={x}, y={y}, dùng={used}, T={T}")
    while used < T and y > 0:
        x, y, used = x + 1, y - 1, used + 2
        print(f"  đổi 1 small -> jumbo: x={x}, y={y}, dùng={used}")
    ok = used == T
    print("kết quả:", [x, y] if ok else [], "(dùng vừa đủ)" if ok else "(không khớp)")

trace_convert(16, 7)
print()
trace_convert(17, 4)

bắt đầu: x=0, y=7, dùng=14, T=16
  đổi 1 small -> jumbo: x=1, y=6, dùng=16
kết quả: [1, 6] (dùng vừa đủ)

bắt đầu: x=0, y=4, dùng=8, T=17
  đổi 1 small -> jumbo: x=1, y=3, dùng=10
  đổi 1 small -> jumbo: x=2, y=2, dùng=12
  đổi 1 small -> jumbo: x=3, y=1, dùng=14
  đổi 1 small -> jumbo: x=4, y=0, dùng=16
kết quả: [] (không khớp)


In [18]:
def trace_binary(T, C):
    lo, hi = 0, C
    step = 0
    while lo < hi:
        mid = (lo + hi) // 2
        f = 2 * C + 2 * mid
        step += 1
        print(f"vòng {step}: lo={lo}, hi={hi}, mid={mid}, f(mid)={f}, f>=T? {f >= T}")
        if f >= T:
            hi = mid
        else:
            lo = mid + 1
    f = 2 * C + 2 * lo
    print(f"dừng: x={lo}, f(x)={f} ->", [lo, C - lo] if f == T else [])

trace_binary(26, 9)
print()
trace_binary(4, 17)

vòng 1: lo=0, hi=9, mid=4, f(mid)=26, f>=T? True
vòng 2: lo=0, hi=4, mid=2, f(mid)=22, f>=T? False
vòng 3: lo=3, hi=4, mid=3, f(mid)=24, f>=T? False
dừng: x=4, f(x)=26 -> [4, 5]

vòng 1: lo=0, hi=17, mid=8, f(mid)=50, f>=T? True
vòng 2: lo=0, hi=8, mid=4, f(mid)=42, f>=T? True
vòng 3: lo=0, hi=4, mid=2, f(mid)=38, f>=T? True
vòng 4: lo=0, hi=2, mid=1, f(mid)=36, f>=T? True
vòng 5: lo=0, hi=1, mid=0, f(mid)=34, f>=T? True
dừng: x=0, f(x)=34 -> []


## 9. Code nộp LeetCode

Dùng công thức đóng (Cách 4): `O(1)`, ngắn và rõ ràng. Chữ ký đúng như skeleton.

In [19]:
class Solution:
    def numOfBurgers(self, tomatoSlices: int, cheeseSlices: int) -> List[int]:
        T, C = tomatoSlices, cheeseSlices
        # có nghiệm <=> T chẵn và 2C <= T <= 4C
        if T % 2 == 1 or T < 2 * C or T > 4 * C:
            return []
        jumbo = T // 2 - C
        small = C - jumbo
        return [jumbo, small]

for t, c, want in examples:
    got = Solution().numOfBurgers(t, c)
    print(got, "OK" if got == want else "SAI")

[1, 6] OK
[] OK
[] OK


## 10. Vì sao thuật toán đúng

1. **Điều kiện cần:** nếu có nghiệm `(x, y)` nguyên không âm thì `T = 4x + 2y = 2(2x + y)` chẵn; `T = 2C + 2x ≥ 2C` (vì `x ≥ 0`); `T = 4C − 2y ≤ 4C` (vì `y ≥ 0`).
2. **Điều kiện đủ:** nếu `T` chẵn và `2C ≤ T ≤ 4C` thì `x = T/2 − C` là số nguyên `≥ 0`, `y = C − x = 2C − T/2 ≥ 0`. Kiểm tra: `x + y = C` và `4x + 2y = 2C + 2x = T`. ✔
3. **Duy nhất:** hệ có định thức khác 0 ⇒ chỉ một nghiệm. Vậy trả đúng nghiệm đó, hoặc `[]` khi điều kiện không thoả.
4. **Binary search đúng** vì `f(x) = 2C + 2x` tăng ngặt trên `[0, C]`; `x` nhỏ nhất với `f(x) ≥ T` là ứng viên duy nhất có thể cho `f(x) = T`.

## 11. Phân tích Time / Space

Gọi `T = tomatoSlices`, `C = cheeseSlices` (đều `≤ 10^7`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn hai ẩn | `O(T²)` | `O(1)` |
| 1b. Vét cạn một ẩn | `O(C)` | `O(1)` |
| 2. Đổi dần (tính trực tiếp) | `O(1)` (mô phỏng: `O(C)`) | `O(1)` |
| 3. Binary search | `O(log C)` ≈ 24 vòng | `O(1)` |
| 4. Công thức | `O(1)` | `O(1)` |

Không có khác biệt average/worst ở các cách `O(1)` và `O(log C)`.

## 12. Bộ test

Đối chiếu mọi cách trên **toàn bộ** lưới `(T, C)` nhỏ (vét cạn hai ẩn làm chuẩn), thêm edge case và test ngẫu nhiên lớn (seed cố định) đối chiếu với vét cạn một ẩn và kiểm tra tính hợp lệ.

In [20]:
ALL = {
    "brute1": burgers_brute1,
    "convert_slow": burgers_convert_slow,
    "convert": burgers_convert,
    "binary": burgers_binary,
    "bisect": burgers_bisect,
    "formula": burgers_formula,
    "SolBinary": lambda t, c: SolutionBinary().numOfBurgers(t, c),
    "SolFormula": lambda t, c: SolutionFormula().numOfBurgers(t, c),
    "Solution": lambda t, c: Solution().numOfBurgers(t, c),
}

fixed = [
    (16, 7, [1, 6]), (17, 4, []), (4, 17, []),
    (0, 0, [0, 0]),          # không nguyên liệu -> 0 burger mỗi loại
    (0, 1, []), (2, 0, []),  # thừa phô mai / thừa cà chua
    (2, 1, [0, 1]), (4, 1, [1, 0]), (6, 1, []),
    (4 * 10**7, 10**7, [10**7, 0]), (2 * 10**7, 10**7, [0, 10**7]),
    (10**7, 10**7, []), (10**7 - 1, 10**6, []),
]
for t, c, want in fixed:
    for name, f in ALL.items():
        got = f(t, c) if name not in ("convert_slow", "brute1") or c <= 10**6 else want
        assert got == want, (name, t, c, got, want)
print("Test cố định: qua", len(fixed), "case")

Test cố định: qua 13 case


In [21]:
# Lưới đầy đủ (T, C) nhỏ, chuẩn là vét cạn hai ẩn
grid = 0
for T in range(0, 81):
    for C in range(0, 41):
        want = burgers_brute2(T, C)
        for name, f in ALL.items():
            assert f(T, C) == want, (name, T, C)
        grid += 1
print("Lưới nhỏ: tất cả", len(ALL), "cách khớp vét cạn trên", grid, "cặp (T, C)")

Lưới nhỏ: tất cả 9 cách khớp vét cạn trên 3321 cặp (T, C)


In [22]:
def is_valid(T, C, ans):
    if ans == []:
        return True       # kiểm tra "không có nghiệm" bằng đối chiếu riêng
    x, y = ans
    return isinstance(x, int) and isinstance(y, int) and x >= 0 and y >= 0 \
        and 4 * x + 2 * y == T and x + y == C

rng = random.Random(1276)
fast = ["binary", "bisect", "formula", "SolBinary", "SolFormula", "Solution", "convert"]
for _ in range(3000):
    C = rng.randint(0, 10**7)
    # một nửa số test cố tình nằm trong đoạn hợp lệ để có nhiều trường hợp có nghiệm
    T = rng.randint(2 * C, 4 * C) if rng.random() < 0.5 else rng.randint(0, 10**7)
    res = {name: ALL[name](T, C) for name in fast}
    first = res["Solution"]
    assert all(r == first for r in res.values()), (T, C, res)
    assert is_valid(T, C, first)
    has = T % 2 == 0 and 2 * C <= T <= 4 * C
    assert (first != []) == has
print("3000 test ngẫu nhiên lớn: các cách nhanh thống nhất và hợp lệ")

3000 test ngẫu nhiên lớn: các cách nhanh thống nhất và hợp lệ


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `/` thay cho `//` và không kiểm tra tính chẵn

Kết quả là số thực, thậm chí là "nửa cái burger".

In [23]:
def wrong_float(T, C):
    x = (T - 2 * C) / 2
    y = (4 * C - T) / 2
    if x < 0 or y < 0:
        return []
    return [x, y]

print("Code sai, (16, 7):", wrong_float(16, 7), "-> kiểu", type(wrong_float(16, 7)[0]).__name__)
print("Code sai, (17, 8):", wrong_float(17, 8), "| đúng là:", Solution().numOfBurgers(17, 8))

Code sai, (16, 7): [1.0, 6.0] -> kiểu float
Code sai, (17, 8): [0.5, 7.5] | đúng là: []


`T = 17` lẻ nên không có nghiệm, nhưng code sai trả `[0.5, 7.5]`. **Code đúng:** kiểm tra `T % 2 == 0` rồi mới dùng `//`.

In [24]:
print("Code đúng:", Solution().numOfBurgers(16, 7), Solution().numOfBurgers(17, 8))

Code đúng: [1, 6] []


### Lỗi 2 — Dùng `//` nhưng quên kiểm tra `T` chẵn

`//` làm tròn xuống, che giấu phần lẻ ⇒ trả về một cặp **không dùng hết** cà chua.

In [25]:
def wrong_no_parity(T, C):
    x = (T - 2 * C) // 2
    y = C - x
    if x < 0 or y < 0:
        return []
    return [x, y]

ans = wrong_no_parity(17, 4)
print("Code sai, (17, 4):", ans, "-> cà chua dùng =", 4 * ans[0] + 2 * ans[1], "≠ 17")
print("Đúng là:", Solution().numOfBurgers(17, 4))

Code sai, (17, 4): [4, 0] -> cà chua dùng = 16 ≠ 17
Đúng là: []


**Code đúng:** thêm điều kiện `T % 2 == 1 ⇒ []` (như `Solution`).

### Lỗi 3 — Quên kiểm tra nghiệm không âm

Công thức luôn cho ra một cặp số, nhưng có thể âm — không thể làm "−15 cái burger".

In [26]:
def wrong_no_sign(T, C):
    if T % 2:
        return []
    x = T // 2 - C
    return [x, C - x]

print("Code sai, (4, 17):", wrong_no_sign(4, 17), "| đúng là:", Solution().numOfBurgers(4, 17))
print("Code sai, (40, 5):", wrong_no_sign(40, 5), "| đúng là:", Solution().numOfBurgers(40, 5))

Code sai, (4, 17): [-15, 32] | đúng là: []
Code sai, (40, 5): [15, -10] | đúng là: []


**Code đúng:** kiểm tra `2C ≤ T ≤ 4C` (tương đương `x ≥ 0` và `y ≥ 0`).

### Lỗi 4 — Vét cạn bằng `range(C)` thay vì `range(C + 1)`

Bỏ sót trường hợp **toàn jumbo** (`x = C`).

In [27]:
def wrong_range(T, C):
    for x in range(C):           # SAI: thiếu x = C
        if 4 * x + 2 * (C - x) == T:
            return [x, C - x]
    return []

print("Code sai, (4, 1):", wrong_range(4, 1), "| đúng là:", Solution().numOfBurgers(4, 1))
print("Code sai, (12, 3):", wrong_range(12, 3), "| đúng là:", Solution().numOfBurgers(12, 3))

Code sai, (4, 1): [] | đúng là: [1, 0]
Code sai, (12, 3): [] | đúng là: [3, 0]


**Code đúng:** `for x in range(C + 1)` (xem `burgers_brute1`). Bài học: luôn tự hỏi "biên trên có được phép xảy ra không?".

### Lỗi 5 — Trả về `None` hoặc `[0, 0]` khi vô nghiệm

Đề yêu cầu `[]`. Trả `[0, 0]` là sai vì `[0, 0]` lại là đáp án **đúng** cho `(0, 0)`.

In [28]:
def wrong_return(T, C):
    if T % 2 or not (2 * C <= T <= 4 * C):
        return [0, 0]            # SAI
    x = T // 2 - C
    return [x, C - x]

print("Code sai, (17, 4):", wrong_return(17, 4), "| đúng là:", Solution().numOfBurgers(17, 4))

Code sai, (17, 4): [0, 0] | đúng là: []


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "**gà và chó**" (đếm chân và đầu) đội lốt burger: hai loại đối tượng, hai đại lượng tổng ⇒ hệ hai phương trình hai ẩn.
- Mẹo nhẩm: "giả sử tất cả là loại nhỏ, phần thừa chia cho chênh lệch" ⇒ `x = (T − 2C) / (4 − 2)`.
- Luôn kiểm tra 3 thứ của nghiệm: **nguyên**, **không âm**, **khớp cả hai phương trình**.
- Khi thấy một đại lượng **tăng đơn điệu** theo ẩn (ở đây `f(x) = 2C + 2x`) ⇒ có thể binary search ngay cả khi chưa nghĩ ra công thức.
- Ràng buộc `10^7` + chủ đề Math ⇒ thường mong đợi `O(1)` hoặc `O(log n)`.

## 15. Biến thể của bài

1. **Tổng quát hoá khẩu phần:** jumbo dùng `a` cà chua, small dùng `b` cà chua (`a ≠ b`), mỗi loại 1 phô mai ⇒ `x = (T − bC) / (a − b)`, cần chia hết và `0 ≤ x ≤ C`.
2. **Mỗi loại dùng số phô mai khác nhau** ⇒ hệ tổng quát `a1·x + b1·y = T`, `a2·x + b2·y = C` — giải bằng quy tắc Cramer, kiểm tra chia hết và dấu.
3. **Ba loại burger** ⇒ hệ thiếu phương trình, có thể có nhiều nghiệm; bài toán chuyển thành **đếm** số nghiệm (duyệt một ẩn, giải hai ẩn còn lại) — xem Bài tập 5.
4. **Cho phép thừa nguyên liệu, tối đa hoá số burger** ⇒ tham lam / quy hoạch tuyến tính nguyên nhỏ.
5. Bài cùng họ: LC 2177 *Find Three Consecutive Integers That Sum to a Given Number*, LC 1716 *Calculate Money in Leetcode Bank*, bài toán "gà và chó" kinh điển.

In [29]:
def burgers_cramer(T, C, a1=4, b1=2, a2=1, b2=1):
    """Giải a1*x + b1*y = T, a2*x + b2*y = C với nghiệm nguyên không âm (Cramer)."""
    det = a1 * b2 - a2 * b1
    if det == 0:
        return None                     # suy biến, cần xử lý riêng
    nx = T * b2 - C * b1
    ny = a1 * C - a2 * T
    if nx % det or ny % det:
        return []
    x, y = nx // det, ny // det
    return [x, y] if x >= 0 and y >= 0 else []

print(burgers_cramer(16, 7), burgers_cramer(17, 4), burgers_cramer(4, 17))
assert all(burgers_cramer(T, C) == Solution().numOfBurgers(T, C) for T in range(60) for C in range(30))
print("Cramer khớp Solution trên lưới nhỏ")

[1, 6] [] []
Cramer khớp Solution trên lưới nhỏ


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn hai ẩn | thử mọi `(x, y)` | `O(T²)` | `O(1)` | làm chuẩn để test, số nhỏ |
| 1b. Vét cạn một ẩn | `y = C − x`, thử `x` | `O(C)` | `O(1)` | khi chưa thấy công thức, `C` vừa phải |
| 2. Đổi dần small → jumbo | thiếu bao nhiêu chia 2 | `O(1)` | `O(1)` | giải thích trực giác, phỏng vấn |
| 3. Binary search | `f(x) = 2C + 2x` đơn điệu | `O(log C)` | `O(1)` | khi hàm đơn điệu nhưng khó giải đóng |
| 4. Công thức đóng | giải hệ, kiểm tra chẵn & dấu | `O(1)` | `O(1)` | **nộp bài** |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** Đoán kết quả của `numOfBurgers(0, 0)`, `numOfBurgers(2, 1)`, `numOfBurgers(6, 2)`, `numOfBurgers(10, 2)`.

**Bài 2 (tính tay):** Với `T = 26, C = 9`, tính tay bằng cách "toàn small rồi đổi dần" và bằng công thức. Kiểm tra lại cả hai phương trình.

**Bài 3 (điền TODO):** Hoàn thiện `burgers_general_todo(T, C, a, b)`: jumbo dùng `a` cà chua, small dùng `b` cà chua (`a > b > 0`), mỗi loại 1 phô mai.

**Bài 4 (tự viết lại):** Không nhìn mục 7, tự viết lại lời giải binary search `burgers_bs_mine(T, C)` theo mẫu "tìm `x` nhỏ nhất thoả điều kiện".

**Bài 5 (mini challenge):** Thêm loại **Mega Burger**: 6 cà chua + 1 phô mai. Viết `count_three_types(T, C)` đếm số bộ `(x, y, z)` nguyên không âm (jumbo, small, mega) dùng hết nguyên liệu. Yêu cầu `O(C)` hoặc tốt hơn.

In [30]:
def burgers_general_todo(T, C, a, b):
    # TODO 1: gọi x = số jumbo; viết phương trình a*x + b*(C - x) = T
    # TODO 2: suy ra x = (T - b*C) / (a - b), kiểm tra chia hết
    # TODO 3: kiểm tra 0 <= x <= C, trả [x, C - x] hoặc []
    pass

def burgers_bs_mine(T, C):
    # TODO: binary search x trong [0, C]
    pass

# Lưu ý: cell này chỉ định nghĩa hàm, không gọi.

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Kiểm tra `T` chẵn. (2) Kiểm tra `2C ≤ T ≤ 4C`. (3) `x = T/2 − C`.

**Bài 2.** (1) Toàn small dùng `18` cà chua. (2) Thiếu `8`, mỗi lần đổi bù `2`. (3) Đổi `4` lần ⇒ `[4, 5]`; kiểm tra `16 + 10 = 26`, `4 + 5 = 9`.

**Bài 3.** (1) Toàn small dùng `b·C`. (2) Mỗi lần đổi thêm `a − b`. (3) `need = T − bC` phải `≥ 0`, chia hết cho `a − b`, và `need // (a − b) ≤ C`.

**Bài 4.** (1) `lo, hi = 0, C`. (2) `mid` thoả `2C + 2·mid ≥ T` ⇒ `hi = mid`, ngược lại `lo = mid + 1`. (3) Sau vòng lặp kiểm tra bằng đúng `T`.

**Bài 5.** (1) Duyệt `z` (số mega) từ `0` đến `C`. (2) Với `z` cố định, còn `T' = T − 6z`, `C' = C − z` ⇒ bài gốc hai loại, có `0` hoặc `1` nghiệm. (3) Cộng dồn; kiểm chứng bằng vét cạn ba vòng lặp với số nhỏ.

## 19. Đáp án tham khảo

In [31]:
# Bài 1
assert Solution().numOfBurgers(0, 0) == [0, 0]
assert Solution().numOfBurgers(2, 1) == [0, 1]
assert Solution().numOfBurgers(6, 2) == [1, 1]
assert Solution().numOfBurgers(10, 2) == []

# Bài 2
assert Solution().numOfBurgers(26, 9) == [4, 5]

# Bài 3
def burgers_general_sol(T, C, a, b):
    need = T - b * C
    if need < 0 or need % (a - b):
        return []
    x = need // (a - b)
    return [x, C - x] if x <= C else []

for T in range(0, 70):
    for C in range(0, 20):
        assert burgers_general_sol(T, C, 4, 2) == Solution().numOfBurgers(T, C)
        brute = [[x, C - x] for x in range(C + 1) if 5 * x + 3 * (C - x) == T]
        assert burgers_general_sol(T, C, 5, 3) == (brute[0] if brute else [])

# Bài 4
def burgers_bs_sol(T, C):
    lo, hi = 0, C
    while lo < hi:
        mid = (lo + hi) // 2
        if 2 * C + 2 * mid >= T:
            hi = mid
        else:
            lo = mid + 1
    return [lo, C - lo] if 2 * C + 2 * lo == T else []

# Bài 5
def count_three_types_sol(T, C):
    total = 0
    for z in range(C + 1):
        T2, C2 = T - 6 * z, C - z
        if T2 < 0:
            break
        if Solution().numOfBurgers(T2, C2):
            total += 1
    return total

def count_three_brute(T, C):
    return sum(1 for x in range(C + 1) for z in range(C + 1 - x)
               if 4 * x + 2 * (C - x - z) + 6 * z == T)

rng = random.Random(7)
for _ in range(2000):
    C = rng.randint(0, 10**6)
    T = rng.randint(0, 5 * 10**6)
    assert burgers_bs_sol(T, C) == Solution().numOfBurgers(T, C)
for T in range(0, 61):
    for C in range(0, 16):
        assert count_three_types_sol(T, C) == count_three_brute(T, C)
print("Ví dụ: count_three_types_sol(24, 6) =", count_three_types_sol(24, 6))
print("Đáp án Bài 1–5: mọi assert đều qua")

Ví dụ: count_three_types_sol(24, 6) = 4
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "hai loại đối tượng, hai tổng":**

```python
# a*x + b*y = T ;  x + y = C   (a > b)
need = T - b * C               # giả sử toàn loại nhỏ
if need < 0 or need % (a - b):
    return []
x = need // (a - b)
if x > C:
    return []
return [x, C - x]
```

**Template binary search trên đáp án (hàm tăng):**

```python
lo, hi = 0, C
while lo < hi:
    mid = (lo + hi) // 2
    if f(mid) >= target: hi = mid
    else: lo = mid + 1
# kiểm tra f(lo) == target
```

**Checklist trước khi submit:**
- [ ] `T` lẻ ⇒ `[]`.
- [ ] `T < 2C` hoặc `T > 4C` ⇒ `[]`.
- [ ] Dùng `//`, không dùng `/` (kết quả phải là `int`).
- [ ] Trường hợp `(0, 0)` trả `[0, 0]`, không phải `[]`.
- [ ] Thứ tự đáp án: `[jumbo, small]`.
- [ ] Đúng chữ ký: `numOfBurgers(self, tomatoSlices: int, cheeseSlices: int) -> List[int]`.